<a href="https://colab.research.google.com/github/aravindanmoorthy/Claude-Hackathon/blob/claude%2Fweather-alerts-parked-cars-58mry/safe_pilot_moving_vehicle_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Safe Pilot — Moving Vehicle Weather Alert System
### USAA Hackathon Project — Phase 2: Dynamic Vehicle Trajectory

Predicts a moving vehicle's future path using GPS telemetry and intersects that path
with live weather data to generate intent-aware, fatigue-free weather alerts.

---

## Architecture Overview

```
GPS Pings (10-15 samples)
        │
        ▼
┌─────────────────────────┐
│  Phase 1: Data Eng.     │  → Speed, Heading, Accel, Map-Match
└────────────┬────────────┘
             │
             ▼
┌─────────────────────────┐
│  Phase 2: LSTM Model    │  → Predicted Path Corridor (20 min)
└────────────┬────────────┘
             │
             ▼
┌─────────────────────────┐
│  Phase 3: NWS + Shapely │  → Spatial Weather Intersection
└────────────┬────────────┘
             │
             ▼
┌─────────────────────────┐
│  Phase 4: RF Decision   │  → Intent-Aware Alert (Red/Yellow)
└─────────────────────────┘
```

**APIs:** [Open-Meteo](https://open-meteo.com/) · [NWS](https://api.weather.gov/) — Free, no key required!

---
**Sections:**
- 📦 **Setup** — Install & imports
- ⚙️ **Phase 1** — GPS feature engineering & map matching
- 🧠 **Phase 2** — LSTM trajectory prediction + probability corridor
- 🌩️ **Phase 3** — NWS weather intersection + nowcasting
- 🎯 **Phase 4** — Random Forest alert decision engine
- 🌍 **Section A** — Live demo with real GPS trace
- 🧪 **Section B** — Simulated moving vehicle scenarios
- ✏️ **Section C** — Test your own GPS trace

> Run each cell top to bottom using **Shift+Enter**

Phase 1 — Data Engineering

smooth_gps() — 3-ping moving average to remove GPS jitter from mobile sensors
extract_features() — Haversine distance → instantaneous speed (mph), atan2 bearing → heading (0–360°), Δspeed/Δtime → acceleration/jerk
map_match() — Speed-based road classification (motorway/trunk/primary/secondary/residential), which mirrors OSMNX output without requiring an API key for the hackathon

Phase 2 — LSTM Trajectory Prediction

_lstm_predict_waypoints() — Physics-based kinematic surrogate (swap the body for a real TensorFlow LSTM in production). Projects the vehicle to +5/+10/+15/+20 min using weighted heading smoothing
_compute_confidence() — Heading stability, road predictability, deceleration penalty, and time-horizon decay all factor into the confidence score
build_corridor() — Unions circular uncertainty zones around each waypoint into a single Shapely polygon corridor. Width scales inversely with confidence (narrow on highways, wide at interchanges)

Phase 3 — Weather Interception

fetch_nws_polygons() — Pulls live GeoJSON warning polygons from api.weather.gov (Tornado, Hail, Flash Flood, etc.)
intersect_corridor_with_weather() — Polygon.intersects() check, then computes overlap fraction to estimate P(impact). Falls back to Open-Meteo hourly forecast if no NWS polygon exists
find_nearest_exit() — Walks backward along waypoints to find the last safe point before the weather zone, returns "Safe exit in ~2.3 miles (~3 min)"

Phase 4 — Intent-Aware Alert Logic

_build_alert_model() — Random Forest trained on 16 labeled synthetic scenarios with 7 features: severity, time-to-impact, P(intersect), road type, braking flag, current weather severity, speed
Red Alert if RF P(red) ≥ 70%, Yellow if combined P ≥ 30%, None otherwise
All advice strings replaced with in-motion driver guidance (e.g. "Exit highway now — hail can crack windshield at speed")

Here's a step-by-step guide to demo your notebook:

---

## 🛡️ Safe Pilot Moving Vehicle — Demo Guide

### Step 1: Open the Notebook in Google Colab

1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Click **File → Upload notebook**
3. Upload the `safe_pilot_moving_vehicle.ipynb` file you just downloaded
4. Once loaded, you'll see all the cells organized by phase

---

### Step 2: Run Setup Cells (top to bottom)

Click **Runtime → Run all** OR manually run each cell with **Shift+Enter** in this order:

| Cell | What it does |
|---|---|
| 📦 Install | Installs `numpy`, `scikit-learn`, `shapely`, `requests` |
| 📦 Imports | Loads all libraries |
| ⚙️ Constants | Loads WMO weather codes, severity levels, advice strings |
| ⚙️ Data Models | Defines `GPSPing`, `PathCorridor`, `MovingVehicleAlert` dataclasses |

You should see green ✅ checkmarks after each cell.

---

### Step 3: Run the Phase Cells

Run each phase cell in order — these define the AI pipeline functions but don't execute the demo yet:

- **Phase 1** cells → GPS smoothing, Haversine speed, bearing, map-matching
- **Phase 2** cells → LSTM surrogate model + corridor builder
- **Phase 3** cells → NWS polygon fetcher + Shapely intersection
- **Phase 4** cells → Random Forest training + alert decision engine
- **Pipeline + Helpers** cells → orchestrator, display formatter, simulation utilities

---

### Step 4: Demo Section A — Live Weather (Best for Judging Panel)

Run the **Section A** cell. This makes real API calls to NWS and Open-Meteo and checks three vehicles driving through:
- Oklahoma City, OK — Tornado Alley, northbound I-35 at 70 mph
- Miami, FL — eastbound I-836 at 55 mph
- Denver, CO — southbound I-25 at 65 mph

**What to point out to judges:**
- The predicted path corridor printed for each vehicle (+5/+10/+15/+20 min waypoints)
- The confidence score — notice it's higher on the highway vehicle
- Whether a live NWS warning is active in those areas that day
- The alert level and driver action advice at the bottom of each output block

---

### Step 5: Demo Section B — Simulated Scenarios (Most Reliable for Demo)

Run the **Section B** cell. This injects synthetic weather polygons so the output is always predictable regardless of real-world weather. Walk through each scenario:

| Scenario | What to show |
|---|---|
| 🔴 Scenario 1 | Tornado Warning polygon placed on the vehicle's predicted highway path — watch the RED alert fire with exit guidance |
| 🔴 Scenario 2 | Thunderstorm + Hail overlapping the NE corridor — RED alert, hail-specific advice |
| 🟡 Scenario 3 | Flash flood zone on a residential road at 20 mph — YELLOW because low speed = lower P(impact) |
| 🟡 Scenario 4 | No NWS polygon, but Open-Meteo forecast shows heavy rain in 15 min — shows the fallback path |
| ✅ Scenario 5 | Clear sky highway — demonstrates the system stays silent (no alert fatigue) |

**Key talking point:** The same storm severity produces RED on a highway and YELLOW on a residential road — the Random Forest is factoring in road type and speed, not just the weather.

---

### Step 6: Demo Section C — Live Custom Location

Scroll to Section C and update these 4 variables before running:

```python
MY_LAT       = 32.9537   # Your actual latitude
MY_LON       = -96.8270  # Your actual longitude (Frisco, TX example)
MY_HEADING   = 0         # Direction you're "driving" (0 = North)
MY_SPEED_MPH = 65        # Speed
LIVE_MODE    = True      # Pull real NWS + Open-Meteo data
```

To get your coordinates: right-click any point on Google Maps → the lat/lon appears at the top of the context menu. This is a great live moment for the judges — run it against the actual city where the hackathon is happening.

---

### Step 7: Key Points to Emphasize in Your Pitch

1. **No destination needed** — the LSTM predicts where the car is going from motion alone
2. **Probability tube, not a line** — the corridor widens at uncertain spots (interchanges), narrows on straight highways
3. **Two data sources cross-checked** — NWS polygons for official warnings + Open-Meteo for nowcasting
4. **Alert fatigue prevention** — the Random Forest blocks yellow/red alerts when probability is below threshold; Scenario 5 shows this explicitly
5. **Exit guidance** — "Safe exit in ~2.3 miles" is actionable, not just a warning
6. **Extensible** — the `_lstm_predict_waypoints()` function is a clearly marked swap point for a real TensorFlow LSTM trained on historical driving data

---

### Troubleshooting

| Problem | Fix |
|---|---|
| NWS API returns empty | Normal when no active warnings in the area — Section B simulates these |
| `ModuleNotFoundError: shapely` | Re-run the install cell, then restart runtime via **Runtime → Restart runtime** |
| Section A shows all NONE | Good weather today! Use Section B for guaranteed alert demo |
| Slow execution | Each Section A vehicle makes 2 API calls — expect 5–10 seconds per vehicle |

---
## 📦 Setup — Install Dependencies & Imports

In [ ]:
# Install all required libraries
# - numpy / scikit-learn : LSTM surrogate + Random Forest decision model
# - shapely              : Geometric corridor/polygon intersection
# - requests             : HTTP calls to Open-Meteo and NWS APIs
!pip install numpy scikit-learn shapely requests --quiet
print('✅ Dependencies ready!')

In [ ]:
import math
import json
import time
import requests
import numpy as np
from copy import deepcopy
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

# Shapely for spatial geometry (corridor ↔ weather polygon intersection)
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import unary_union

# Scikit-learn used as lightweight LSTM surrogate + Random Forest
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

print('✅ Imports successful!')

---
## ⚙️ Constants & Data Models

In [ ]:
# ─────────────────────────────────────────────────────────────────
# WMO WEATHER CODES (same as original Safe Pilot)
# ─────────────────────────────────────────────────────────────────
ALERT_THRESHOLD_CODES = {95, 96, 99, 82, 75, 65, 45, 48}

SEVERE_WEATHER_CODES = {
    0:  'Clear Sky',       1:  'Mainly Clear',    2:  'Partly Cloudy',
    3:  'Overcast',        45: 'Foggy',            48: 'Icy Fog',
    51: 'Light Drizzle',   61: 'Light Rain',       63: 'Moderate Rain',
    65: 'Heavy Rain',      71: 'Light Snow',       73: 'Moderate Snow',
    75: 'Heavy Snow',      77: 'Snow Grains',      80: 'Rain Showers',
    81: 'Heavy Showers',   82: 'Violent Showers',  85: 'Snow Showers',
    95: 'Thunderstorm',    96: 'Thunderstorm + Hail', 99: 'Thunderstorm + Heavy Hail',
}

ALERT_SEVERITY = {
    45: 'MEDIUM', 48: 'HIGH',   65: 'MEDIUM',
    75: 'HIGH',   82: 'HIGH',   95: 'HIGH',
    96: 'HIGH',   99: 'CRITICAL',
}

# ─────────────────────────────────────────────────────────────────
# MOVING VEHICLE ADVICE (replaces parked-vehicle advice)
# Actionable in-motion guidance prioritizing driver safety.
# ─────────────────────────────────────────────────────────────────
MOVING_VEHICLE_ADVICE = {
    45: 'Reduce speed — low visibility fog ahead. Use fog lights.',
    48: 'Black ice risk. Reduce speed significantly, increase following distance.',
    65: 'Heavy rain ahead. Slow down and watch for standing water.',
    75: 'Heavy snow on predicted route. Seek shelter or alternate route now.',
    82: 'Flash flood risk on path. Avoid low-lying roads ahead.',
    95: 'Thunderstorm on predicted path. Seek shelter before it arrives.',
    96: 'Hail storm ahead! Exit highway now — hail can crack windshield at speed.',
    99: 'CRITICAL: Severe hail storm directly on your path. Pull over under cover NOW.',
}

# Road type classifications (from OSMNX / map-matching)
ROAD_TYPES = {
    'motorway':     {'speed_limit': 70, 'predictability': 0.95, 'shelter_access': 0.2},
    'trunk':        {'speed_limit': 55, 'predictability': 0.90, 'shelter_access': 0.4},
    'primary':      {'speed_limit': 45, 'predictability': 0.80, 'shelter_access': 0.6},
    'secondary':    {'speed_limit': 35, 'predictability': 0.70, 'shelter_access': 0.7},
    'residential':  {'speed_limit': 25, 'predictability': 0.50, 'shelter_access': 0.9},
    'unknown':      {'speed_limit': 35, 'predictability': 0.60, 'shelter_access': 0.5},
}

print('✅ Constants loaded!')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DATA MODELS
# ─────────────────────────────────────────────────────────────────

@dataclass
class GPSPing:
    """A single GPS telemetry sample from the mobile device."""
    lat:       float          # WGS-84 latitude
    lon:       float          # WGS-84 longitude
    alt:       float          # Altitude in meters
    timestamp: datetime       # UTC timestamp of the ping
    speed_mph: float = 0.0    # Computed instantaneous speed (filled by Phase 1)
    heading:   float = 0.0    # Bearing angle 0°–360° relative to North
    accel:     float = 0.0    # Acceleration mph/s (+ = speeding up, - = slowing)
    road_type: str  = 'unknown'  # Map-matched road classification

@dataclass
class PredictedWaypoint:
    """A single AI-predicted future vehicle position."""
    lat:              float   # Predicted latitude
    lon:              float   # Predicted longitude
    minutes_ahead:    int     # How far in the future (5, 10, 15, 20 min)
    confidence:       float   # 0.0–1.0 confidence score from LSTM
    corridor_radius_m: float  # Uncertainty radius in meters (wider = less sure)

@dataclass
class PathCorridor:
    """The probability tube the vehicle is likely to travel through."""
    waypoints:        List[PredictedWaypoint]
    polygon:          object   # Shapely Polygon representing the full corridor
    road_type:        str      # Dominant road type on this corridor
    avg_confidence:   float    # Mean confidence across all waypoints

@dataclass
class WeatherIntersection:
    """Result of intersecting the path corridor with a weather polygon."""
    intersects:       bool
    weather_type:     str
    severity_code:    int
    minutes_to_impact: float   # Time until vehicle reaches the weather zone
    intersection_prob: float   # 0.0–1.0 probability of actual intersection
    exit_before_impact: Optional[str] = None  # e.g. "Exit 104 in 2.1 miles"

@dataclass
class MovingVehicleAlert:
    """Final alert object — analogous to WeatherAlert but for a moving vehicle."""
    # Location state
    current_lat:      float
    current_lon:      float
    speed_mph:        float
    heading_degrees:  float
    road_type:        str
    # Prediction
    corridor:         PathCorridor
    # Weather
    intersection:     Optional[WeatherIntersection]
    current_weather:  str
    current_code:     int
    wind_speed_mph:   float
    temperature_f:    float
    # Decision output
    alert_level:      str    # 'NONE' / 'YELLOW' / 'RED'
    alert_message:    str
    vehicle_advice:   str
    exit_tip:         Optional[str]
    confidence_pct:   float  # % probability of impact that triggered the alert

print('✅ Data models defined!')

---
## ⚙️ Phase 1 — Data Engineering & Feature Extraction

Cleans noisy GPS data and computes vector features:
- **Haversine distance** between pings → instantaneous speed
- **Bearing** calculation → heading (0–360°)
- **Acceleration/jerk** → driver intent (braking = potential hazard ahead)
- **Map-matching** → road type classification (highway vs residential)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 1A — GEOSPATIAL MATH UTILITIES
# ─────────────────────────────────────────────────────────────────

EARTH_RADIUS_M = 6_371_000  # Mean Earth radius in meters
MPH_PER_MS = 2.23694        # 1 m/s = 2.23694 mph

def haversine_distance_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Compute the great-circle distance in meters between two GPS coordinates
    using the Haversine formula.  Accurate to ~0.5% for distances < 500 km.

    Args:
        lat1, lon1: Origin coordinate (degrees)
        lat2, lon2: Destination coordinate (degrees)
    Returns:
        Distance in meters.
    """
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * EARTH_RADIUS_M * math.asin(math.sqrt(a))


def compute_bearing(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Compute the initial compass bearing (0–360°, clockwise from North)
    when travelling from point 1 to point 2.

    Returns:
        Bearing in degrees [0, 360).
    """
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dlam = math.radians(lon2 - lon1)
    x = math.sin(dlam) * math.cos(phi2)
    y = math.cos(phi1)*math.sin(phi2) - math.sin(phi1)*math.cos(phi2)*math.cos(dlam)
    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360


def project_point(lat: float, lon: float, bearing_deg: float, dist_m: float) -> Tuple[float, float]:
    """
    Project a GPS point forward by `dist_m` metres in the direction of `bearing_deg`.
    Uses spherical Earth approximation — accurate enough for < 50 km projections.

    Returns:
        (new_lat, new_lon) in decimal degrees.
    """
    lat_r = math.radians(lat)
    lon_r = math.radians(lon)
    brng  = math.radians(bearing_deg)
    dr    = dist_m / EARTH_RADIUS_M  # angular distance in radians
    lat2  = math.asin(math.sin(lat_r)*math.cos(dr) + math.cos(lat_r)*math.sin(dr)*math.cos(brng))
    lon2  = lon_r + math.atan2(
        math.sin(brng)*math.sin(dr)*math.cos(lat_r),
        math.cos(dr) - math.sin(lat_r)*math.sin(lat2)
    )
    return math.degrees(lat2), math.degrees(lon2)


print('✅ Phase 1A — Geo utilities defined!')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 1B — FEATURE EXTRACTION FROM GPS PING WINDOW
# ─────────────────────────────────────────────────────────────────

def extract_features(pings: List[GPSPing]) -> List[GPSPing]:
    """
    Process a rolling window of 10–15 GPS pings and compute:
      - Instantaneous speed (mph) via Haversine distance / Δtime
      - Heading (bearing angle, 0–360°)
      - Acceleration (Δspeed / Δtime in mph/s)

    The first ping cannot have a speed/heading computed (no prior reference),
    so it inherits the second ping's values for continuity.

    Args:
        pings: Ordered list of GPSPing objects (oldest → newest).
    Returns:
        The same list with speed, heading, and accel fields populated.
    """
    if len(pings) < 2:
        raise ValueError('Need at least 2 GPS pings to compute features.')

    for i in range(1, len(pings)):
        prev, curr = pings[i-1], pings[i]
        dt_s = (curr.timestamp - prev.timestamp).total_seconds()

        if dt_s <= 0:
            # Duplicate or out-of-order timestamps — preserve previous values
            curr.speed_mph = prev.speed_mph
            curr.heading   = prev.heading
            curr.accel     = 0.0
            continue

        # Haversine distance in metres, then convert to speed in mph
        dist_m        = haversine_distance_m(prev.lat, prev.lon, curr.lat, curr.lon)
        speed_ms      = dist_m / dt_s           # m/s
        curr.speed_mph = speed_ms * MPH_PER_MS  # mph

        # Bearing angle (compass heading)
        curr.heading = compute_bearing(prev.lat, prev.lon, curr.lat, curr.lon)

        # Acceleration: change in speed divided by time (mph/s)
        curr.accel = (curr.speed_mph - prev.speed_mph) / dt_s

    # Back-fill first ping from second
    pings[0].speed_mph = pings[1].speed_mph
    pings[0].heading   = pings[1].heading
    pings[0].accel     = 0.0

    return pings


def map_match(pings: List[GPSPing]) -> List[GPSPing]:
    """
    Classify each GPS ping to the most likely road type based on speed heuristics.
    In production this would call OSMNX or the Google Roads API.  Here we use a
    speed-based proxy that closely mirrors real road categories.

    Speed → Road Type mapping:
        ≥ 60 mph  → motorway  (interstate/highway)
        45–60 mph → trunk     (US highway)
        30–45 mph → primary   (arterial)
        20–30 mph → secondary (collector)
        < 20 mph  → residential

    Args:
        pings: List of GPSPing with speed_mph already populated.
    Returns:
        The same list with road_type set on each ping.
    """
    for ping in pings:
        spd = ping.speed_mph
        if spd >= 60:
            ping.road_type = 'motorway'
        elif spd >= 45:
            ping.road_type = 'trunk'
        elif spd >= 30:
            ping.road_type = 'primary'
        elif spd >= 20:
            ping.road_type = 'secondary'
        else:
            ping.road_type = 'residential'
    return pings


def smooth_gps(pings: List[GPSPing], window: int = 3) -> List[GPSPing]:
    """
    Apply a simple moving-average smoothing to lat/lon to reduce GPS jitter.
    Mobile GPS can jump ±10–30 m between pings even when stationary.

    Args:
        pings:  Raw GPS pings.
        window: Rolling average window size (default 3).
    Returns:
        Smoothed ping list.
    """
    lats = [p.lat for p in pings]
    lons = [p.lon for p in pings]

    for i in range(len(pings)):
        lo = max(0, i - window // 2)
        hi = min(len(pings), i + window // 2 + 1)
        pings[i].lat = sum(lats[lo:hi]) / (hi - lo)
        pings[i].lon = sum(lons[lo:hi]) / (hi - lo)

    return pings


def run_phase1(raw_pings: List[GPSPing]) -> List[GPSPing]:
    """Full Phase 1 pipeline: smooth → feature extract → map match."""
    pings = smooth_gps(raw_pings)
    pings = extract_features(pings)
    pings = map_match(pings)
    return pings


print('✅ Phase 1B — Feature extraction defined!')

---
## 🧠 Phase 2 — Trajectory Prediction Model

**LSTM Architecture (production)**  
Input: last 10 vectors of `[Lat, Lon, Heading, Speed]`  
Output: 4 predicted coordinates at +5, +10, +15, +20 minutes

**Implementation Note:**  
A full LSTM requires TensorFlow/PyTorch and pre-training data.  
This notebook uses a **physics-based kinematic model** as the LSTM surrogate —
it projects the vehicle forward using its current speed + heading with
heading-smoothing to follow road curves.  In production, swap `_lstm_predict_waypoints()`
with a call to your trained TensorFlow model.

The **corridor width** (uncertainty radius) widens when:
- Confidence is low (e.g. complex interchange)
- Road predictability score is low (e.g. residential grid)
- The driver is decelerating (possible turn)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 2A — LSTM TRAJECTORY MODEL (Kinematic Surrogate)
# ─────────────────────────────────────────────────────────────────

PREDICTION_HORIZONS_MIN = [5, 10, 15, 20]  # Waypoints to predict
BASE_CORRIDOR_M = 150  # Minimum corridor half-width in metres (~2 lanes)
MAX_CORRIDOR_M  = 800  # Maximum corridor half-width (major interchange uncertainty)


def _compute_confidence(pings: List[GPSPing], horizon_min: int) -> float:
    """
    Estimate trajectory prediction confidence for a future horizon.

    Confidence degrades with:
      - Distance into the future (further = less certain)
      - Heading variability over recent pings (turning = uncertain)
      - Low road predictability score (residential grids are unpredictable)
      - Strong deceleration (driver may be approaching a turn or stop)

    Returns:
        Confidence score [0.0, 1.0] where 1.0 = perfectly certain.
    """
    # --- Heading stability (std dev of recent headings) ---
    headings = [p.heading for p in pings[-5:]]  # Last 5 pings
    if len(headings) > 1:
        # Handle wrap-around at 360/0 degrees
        heading_sin = [math.sin(math.radians(h)) for h in headings]
        heading_cos = [math.cos(math.radians(h)) for h in headings]
        heading_std = math.degrees(math.atan2(
            np.std(heading_sin), np.std(heading_cos)
        ))
        heading_confidence = max(0.3, 1.0 - heading_std / 90)  # 90° turn = 0.3 conf
    else:
        heading_confidence = 0.7

    # --- Road predictability ---
    road_type   = pings[-1].road_type
    road_conf   = ROAD_TYPES.get(road_type, ROAD_TYPES['unknown'])['predictability']

    # --- Deceleration penalty ---
    avg_accel   = sum(p.accel for p in pings[-3:]) / 3
    accel_penalty = min(0.3, abs(min(0, avg_accel)) / 10)  # Braking reduces confidence

    # --- Time horizon decay ---
    horizon_decay = max(0.4, 1.0 - (horizon_min / 20) * 0.45)  # 20 min = 0.55 decay

    confidence = heading_confidence * road_conf * horizon_decay - accel_penalty
    return max(0.1, min(1.0, confidence))


def _lstm_predict_waypoints(pings: List[GPSPing]) -> List[PredictedWaypoint]:
    """
    LSTM surrogate: predict future vehicle positions using kinematic projection.

    In production: replace this function body with a call to a trained
    TensorFlow/PyTorch LSTM model that ingests [Lat, Lon, Heading, Speed]
    sequences and outputs future coordinates.

    Current implementation:
      - Uses average speed from the last 5 pings
      - Projects along the smoothed heading with gentle turn damping
      - Corridor radius = f(confidence, road_type)

    Args:
        pings: Feature-enriched GPS pings from Phase 1.
    Returns:
        List of PredictedWaypoint for each horizon in PREDICTION_HORIZONS_MIN.
    """
    if not pings:
        return []

    # Use last 5 pings for averaging (reduces noise)
    recent = pings[-5:]
    avg_speed_mph = sum(p.speed_mph for p in recent) / len(recent)
    avg_speed_ms  = avg_speed_mph / MPH_PER_MS

    # Smoothed heading: weighted toward most recent ping
    weights  = [0.1, 0.15, 0.2, 0.25, 0.3][-len(recent):]
    headings = [p.heading for p in recent]
    # Vector averaging for wrap-around robustness
    sin_avg = sum(w * math.sin(math.radians(h)) for w, h in zip(weights, headings))
    cos_avg = sum(w * math.cos(math.radians(h)) for w, h in zip(weights, headings))
    smoothed_heading = (math.degrees(math.atan2(sin_avg, cos_avg)) + 360) % 360

    # Starting point = most recent ping
    start_lat = pings[-1].lat
    start_lon = pings[-1].lon
    road_type = pings[-1].road_type

    waypoints = []
    for horizon_min in PREDICTION_HORIZONS_MIN:
        # Distance vehicle will travel in `horizon_min` minutes
        dist_m    = avg_speed_ms * horizon_min * 60
        pred_lat, pred_lon = project_point(start_lat, start_lon, smoothed_heading, dist_m)

        # Confidence score for this horizon
        confidence = _compute_confidence(pings, horizon_min)

        # Corridor radius: wider when uncertain, minimum capped at BASE_CORRIDOR_M
        base_radius = ROAD_TYPES.get(road_type, ROAD_TYPES['unknown'])['speed_limit'] * 2
        corridor_radius = BASE_CORRIDOR_M + (1 - confidence) * (MAX_CORRIDOR_M - BASE_CORRIDOR_M)
        corridor_radius = max(BASE_CORRIDOR_M, min(MAX_CORRIDOR_M, corridor_radius))

        waypoints.append(PredictedWaypoint(
            lat=pred_lat,
            lon=pred_lon,
            minutes_ahead=horizon_min,
            confidence=round(confidence, 3),
            corridor_radius_m=round(corridor_radius, 1),
        ))

    return waypoints


print('✅ Phase 2A — LSTM surrogate model defined!')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 2B — PROBABILITY CORRIDOR GENERATION
# ─────────────────────────────────────────────────────────────────

DEG_PER_METER_LAT = 1 / 111_320   # ~1 degree lat = 111.32 km

def _meters_to_degrees(meters: float, lat: float) -> Tuple[float, float]:
    """Convert a radius in metres to (delta_lat_deg, delta_lon_deg)."""
    dlat = meters * DEG_PER_METER_LAT
    dlon = meters / (111_320 * math.cos(math.radians(lat)))
    return dlat, dlon


def _waypoint_to_circle(wp: PredictedWaypoint) -> Polygon:
    """
    Approximate a circular uncertainty zone around a predicted waypoint
    as a polygon (32-sided) in lat/lon space.

    Args:
        wp: Predicted waypoint with corridor_radius_m set.
    Returns:
        Shapely Polygon representing the uncertainty circle.
    """
    dlat, dlon = _meters_to_degrees(wp.corridor_radius_m, wp.lat)
    n_pts = 32  # Number of polygon vertices (higher = smoother circle)
    pts = [
        (
            wp.lon + dlon * math.cos(2 * math.pi * i / n_pts),
            wp.lat + dlat * math.sin(2 * math.pi * i / n_pts),
        )
        for i in range(n_pts)
    ]
    return Polygon(pts)


def build_corridor(pings: List[GPSPing], waypoints: List[PredictedWaypoint]) -> PathCorridor:
    """
    Construct the full probability corridor polygon by:
      1. Building a circle of uncertainty around each predicted waypoint
      2. Taking the union of all circles → forms a tube along the predicted path
      3. Adding the vehicle's current position as the starting circle

    The corridor is wider at later horizons (where confidence is lower)
    and narrower near the vehicle's current position.

    Args:
        pings:     Feature-enriched GPS pings (for current position).
        waypoints: Predicted future positions from the LSTM model.
    Returns:
        PathCorridor with Shapely polygon and metadata.
    """
    circles = []

    # Start circle: tight radius at current position
    curr_wp = PredictedWaypoint(
        lat=pings[-1].lat,
        lon=pings[-1].lon,
        minutes_ahead=0,
        confidence=1.0,
        corridor_radius_m=BASE_CORRIDOR_M,
    )
    circles.append(_waypoint_to_circle(curr_wp))

    # Future waypoint circles
    for wp in waypoints:
        circles.append(_waypoint_to_circle(wp))

    # Union all circles into one connected corridor polygon
    corridor_polygon = unary_union(circles)

    avg_confidence = sum(wp.confidence for wp in waypoints) / len(waypoints)
    road_type = pings[-1].road_type

    return PathCorridor(
        waypoints=waypoints,
        polygon=corridor_polygon,
        road_type=road_type,
        avg_confidence=round(avg_confidence, 3),
    )


def run_phase2(pings: List[GPSPing]) -> PathCorridor:
    """Full Phase 2 pipeline: LSTM prediction → corridor polygon."""
    waypoints = _lstm_predict_waypoints(pings)
    corridor  = build_corridor(pings, waypoints)
    return corridor


print('✅ Phase 2B — Corridor builder defined!')

---
## 🌩️ Phase 3 — Weather Interception & Nowcasting

Pulls live data from two APIs:
1. **NWS Alerts API** — GeoJSON polygons for active Tornado/Hail/Flood Warnings
2. **Open-Meteo** — Current conditions at vehicle position

Then uses **Shapely** geometry to check if the predicted path corridor
intersects any weather warning polygons.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 3A — NWS WEATHER ALERT POLYGON FETCHER
# ─────────────────────────────────────────────────────────────────

NWS_ALERTS_URL = 'https://api.weather.gov/alerts/active'

# NWS event types → WMO-like severity codes for our alert system
NWS_EVENT_TO_CODE = {
    'Tornado Warning':           99,
    'Tornado Watch':             95,
    'Severe Thunderstorm Warning': 96,
    'Severe Thunderstorm Watch': 95,
    'Flash Flood Warning':       82,
    'Flash Flood Watch':         65,
    'Winter Storm Warning':      75,
    'Ice Storm Warning':         48,
    'Dense Fog Advisory':        45,
    'High Wind Warning':         95,
}


def fetch_nws_polygons(lat: float, lon: float, radius_km: float = 100) -> List[Dict]:
    """
    Fetch active NWS weather alert polygons within `radius_km` of the vehicle.

    Calls the NWS /alerts/active endpoint filtered by a point coordinate.
    Returns a list of dicts, each containing:
      - 'event'    : Alert event name (e.g. 'Tornado Warning')
      - 'severity' : NWS severity string ('Extreme', 'Severe', 'Moderate', etc.)
      - 'polygon'  : Shapely Polygon from the GeoJSON geometry
      - 'code'     : Mapped WMO-style severity code

    Args:
        lat, lon:   Vehicle's current position.
        radius_km:  Search radius in km (NWS API uses point-based queries).
    Returns:
        List of active weather alert polygons (may be empty if all-clear).
    """
    try:
        # NWS uses point query — fetch alerts affecting the general area
        params = {'point': f'{lat:.4f},{lon:.4f}', 'status': 'actual', 'limit': 50}
        headers = {'User-Agent': 'SafePilotWeatherAlert/1.0 (USAA Hackathon)'}
        resp = requests.get(NWS_ALERTS_URL, params=params, headers=headers, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f'  ⚠️  NWS API call failed: {e} — continuing without NWS polygons.')
        return []

    polygons = []
    for feature in data.get('features', []):
        props = feature.get('properties', {})
        event = props.get('event', 'Unknown')
        geom  = feature.get('geometry')

        if geom is None:
            continue  # Some alerts have no polygon (county-based only)

        # Parse GeoJSON geometry into a Shapely object
        geom_type = geom.get('type', '')
        coords    = geom.get('coordinates', [])

        try:
            if geom_type == 'Polygon' and coords:
                # GeoJSON coords are [lon, lat] — Shapely uses (lon, lat) natively
                shapely_polygon = Polygon(coords[0])
            elif geom_type == 'MultiPolygon' and coords:
                shapely_polygon = unary_union([Polygon(ring[0]) for ring in coords])
            else:
                continue
        except Exception:
            continue

        code = NWS_EVENT_TO_CODE.get(event, 95)
        polygons.append({
            'event':    event,
            'severity': props.get('severity', 'Unknown'),
            'polygon':  shapely_polygon,
            'code':     code,
        })

    return polygons


def fetch_weather_at_point(lat: float, lon: float) -> Optional[Dict]:
    """
    Fetch current weather conditions at the vehicle's position via Open-Meteo.
    (Reused from the original Safe Pilot notebook.)

    Returns:
        Raw Open-Meteo API response dict, or None if the request fails.
    """
    url    = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'current': [
            'temperature_2m', 'wind_speed_10m',
            'weather_code',   'precipitation', 'visibility',
        ],
        'hourly': [
            'precipitation_probability', 'wind_gusts_10m',
            'weather_code', 'visibility',
        ],
        'temperature_unit': 'fahrenheit',
        'wind_speed_unit':  'mph',
        'forecast_days': 1,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f'  ⚠️  Open-Meteo API failed: {e}')
        return None


print('✅ Phase 3A — NWS polygon fetcher defined!')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 3B — SPATIAL WEATHER INTERSECTION
# ─────────────────────────────────────────────────────────────────

def find_nearest_exit(corridor: PathCorridor, weather_polygon: Polygon,
                      current_lat: float, current_lon: float,
                      speed_mph: float) -> Optional[str]:
    """
    Estimate the last safe exit point before the vehicle enters the weather zone.

    Walks backward along the predicted corridor waypoints to find the last
    waypoint that does NOT intersect the weather polygon, then estimates
    the distance from the vehicle to that waypoint.

    Args:
        corridor:        Predicted path corridor.
        weather_polygon: Shapely polygon of the weather warning zone.
        current_lat/lon: Vehicle's current position.
        speed_mph:       Vehicle speed (for ETA estimation).
    Returns:
        Human-readable string like 'Safe exit in ~2.3 miles (~3 min)' or None.
    """
    last_safe_wp = None
    for wp in corridor.waypoints:
        wp_point = Point(wp.lon, wp.lat)  # Shapely uses (lon, lat)
        if not weather_polygon.contains(wp_point):
            last_safe_wp = wp
        else:
            break  # Found the first dangerous waypoint — stop here

    if last_safe_wp is None:
        return None  # Vehicle is already inside the weather zone

    # Distance from current position to the last safe waypoint
    dist_m    = haversine_distance_m(current_lat, current_lon,
                                     last_safe_wp.lat, last_safe_wp.lon)
    dist_mi   = dist_m / 1609
    eta_min   = (dist_m / (speed_mph / MPH_PER_MS)) / 60 if speed_mph > 0 else last_safe_wp.minutes_ahead
    return f'Safe exit in ~{dist_mi:.1f} miles (~{int(eta_min)} min)'


def intersect_corridor_with_weather(
    corridor: PathCorridor,
    nws_polygons: List[Dict],
    weather_response: Optional[Dict],
    pings: List[GPSPing],
) -> Optional[WeatherIntersection]:
    """
    Check if the predicted path corridor intersects any active NWS warning polygon.

    Intersection probability accounts for:
      - Whether the corridor polygon geometrically intersects (binary check)
      - The overlap fraction of the corridor inside the weather zone
      - The corridor's average confidence score

    If no NWS polygon is found but Open-Meteo reports severe weather at any
    predicted waypoint, a synthetic intersection is created.

    Args:
        corridor:         Predicted path corridor from Phase 2.
        nws_polygons:     Active NWS warning polygons from Phase 3A.
        weather_response: Open-Meteo API response for the current position.
        pings:            GPS ping history (for current position and speed).
    Returns:
        WeatherIntersection if a threat is detected, else None.
    """
    current_lat   = pings[-1].lat
    current_lon   = pings[-1].lon
    current_speed = pings[-1].speed_mph

    # ── Check NWS polygon intersections first (highest accuracy) ──
    worst_intersection = None
    worst_code = 0

    for alert in nws_polygons:
        wx_polygon = alert['polygon']
        if not corridor.polygon.intersects(wx_polygon):
            continue  # No geometric intersection — skip

        # Compute overlap fraction → intersection probability
        try:
            overlap_area   = corridor.polygon.intersection(wx_polygon).area
            corridor_area  = corridor.polygon.area
            overlap_frac   = overlap_area / corridor_area if corridor_area > 0 else 0.5
        except Exception:
            overlap_frac   = 0.5

        # P(impact) = geometric overlap × corridor confidence
        p_impact = overlap_frac * corridor.avg_confidence
        p_impact = max(0.1, min(1.0, p_impact))

        # Find which predicted waypoint first enters the danger zone
        minutes_to_impact = 0
        for wp in corridor.waypoints:
            wp_pt = Point(wp.lon, wp.lat)
            if wx_polygon.contains(wp_pt) or wp_pt.distance(wx_polygon) < 0.001:
                minutes_to_impact = wp.minutes_ahead
                break

        exit_tip = find_nearest_exit(corridor, wx_polygon,
                                     current_lat, current_lon, current_speed)

        if alert['code'] > worst_code:
            worst_code = alert['code']
            worst_intersection = WeatherIntersection(
                intersects=True,
                weather_type=alert['event'],
                severity_code=alert['code'],
                minutes_to_impact=minutes_to_impact,
                intersection_prob=round(p_impact, 3),
                exit_before_impact=exit_tip,
            )

    if worst_intersection:
        return worst_intersection

    # ── Fallback: check Open-Meteo forecast at future waypoints ──
    # For each predicted waypoint, fetch weather and check severity.
    # This catches storms not yet in the NWS warning system.
    if weather_response:
        hourly_codes = weather_response.get('hourly', {}).get('weather_code', [])
        for wp in corridor.waypoints:
            hour_idx = min(wp.minutes_ahead // 60, len(hourly_codes) - 1)
            if hour_idx >= 0 and hourly_codes[hour_idx] in ALERT_THRESHOLD_CODES:
                # Estimate probability from corridor confidence
                p_impact = corridor.avg_confidence * 0.6  # Moderate confidence from forecast
                return WeatherIntersection(
                    intersects=True,
                    weather_type=SEVERE_WEATHER_CODES.get(hourly_codes[hour_idx], 'Severe Weather'),
                    severity_code=hourly_codes[hour_idx],
                    minutes_to_impact=wp.minutes_ahead,
                    intersection_prob=round(p_impact, 3),
                    exit_before_impact=None,
                )

    return None  # No weather threat detected


def run_phase3(pings: List[GPSPing], corridor: PathCorridor) -> Tuple[Optional[WeatherIntersection], Optional[Dict]]:
    """Full Phase 3 pipeline: fetch weather data → spatial intersection."""
    lat, lon = pings[-1].lat, pings[-1].lon

    # Fetch in parallel conceptually (sequential here for simplicity)
    nws_polygons     = fetch_nws_polygons(lat, lon)
    weather_response = fetch_weather_at_point(lat, lon)

    intersection = intersect_corridor_with_weather(
        corridor, nws_polygons, weather_response, pings
    )
    return intersection, weather_response


print('✅ Phase 3B — Spatial intersection defined!')

---
## 🎯 Phase 4 — Intent-Aware Alert Decision Engine

A **Random Forest classifier** that decides whether to fire an alert and at what level.
This prevents **alert fatigue** by considering:

| Feature | Explanation |
|---|---|
| `weather_severity`       | Numeric severity of the threat (0–4) |
| `time_to_impact_min`     | Minutes until vehicle enters the weather zone |
| `intersection_prob`      | P(vehicle actually hits the weather) |
| `road_type_score`        | Highway (high) vs residential (low) |
| `driver_is_braking`      | Is the driver already reacting? |
| `current_weather_severe` | Is it already bad at the vehicle's current location? |

**Output:**
- **P > 70%** → 🔴 RED Alert
- **30% ≤ P ≤ 70%** → 🟡 YELLOW Advisory
- **P < 30%** → ✅ No Alert

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 4A — RANDOM FOREST ALERT DECISION MODEL  (FIXED)
# ─────────────────────────────────────────────────────────────────

def _build_alert_model() -> RandomForestClassifier:
    # Feature columns: [severity_score, time_to_impact_norm, p_intersect,
    #  road_score, is_braking, current_severe, speed_norm]
    # Labels: 0=No Alert, 1=Yellow, 2=Red
    #
    # FIX: Added 3 RED examples for high-severity events at moderate p_int
    # (0.35-0.50) and highway speeds. Previously these fell into a training-data
    # gap and were misclassified as YELLOW.
    X_train = np.array([
        # severity  time_norm  p_int  road  brake  curr_sev  spd_norm
        [4.0,       0.1,       0.90,  0.9,  0,     0,        0.9],    # Red: tornado, imminent, highway
        [4.0,       0.3,       0.80,  0.9,  0,     0,        0.8],    # Red: tornado, 6 min, highway
        [4.0,       0.5,       0.50,  0.95, 0,     0,        0.93],   # Red: tornado 10min, 70mph  <- ADDED
        [4.0,       0.5,       0.35,  0.90, 0,     0,        0.85],   # Red: tornado 10min, mod prob <- ADDED
        [3.0,       0.2,       0.85,  0.8,  0,     0,        0.7],    # Red: hail, near, trunk
        [3.0,       0.5,       0.75,  0.7,  1,     0,        0.5],    # Red: hail, braking
        [3.0,       0.4,       0.45,  0.90, 0,     0,        0.80],   # Red: hail, 8min, highway   <- ADDED
        [4.0,       0.8,       0.72,  0.5,  0,     0,        0.3],    # Red: high sev, far, high prob
        [2.0,       0.2,       0.60,  0.7,  0,     1,        0.6],    # Red: severe now + incoming
        [3.0,       0.4,       0.65,  0.6,  0,     0,        0.5],    # Yellow: hail, moderate
        [2.0,       0.5,       0.55,  0.7,  0,     0,        0.6],    # Yellow: thunderstorm
        [2.0,       0.6,       0.45,  0.8,  0,     0,        0.7],    # Yellow: rain, 12min
        [1.0,       0.3,       0.50,  0.5,  0,     0,        0.4],    # Yellow: fog, residential
        [3.0,       0.9,       0.35,  0.9,  0,     0,        0.8],    # Yellow: severe but far
        [1.0,       0.8,       0.20,  0.6,  0,     0,        0.5],    # None: low prob, mild
        [0.0,       1.0,       0.05,  0.9,  0,     0,        0.8],    # None: clear highway
        [1.0,       1.0,       0.10,  0.5,  0,     0,        0.3],    # None: mild, far
        [0.0,       0.5,       0.00,  0.8,  0,     0,        0.6],    # None: no intersection
        [2.0,       0.7,       0.28,  0.3,  0,     0,        0.2],    # None: residential
    ])
    # 9 RED, 5 YELLOW, 5 NONE (was 6 RED before fix)
    y_train = np.array([2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    return model


ALERT_MODEL = _build_alert_model()
SEVERITY_NUMERIC = {0: 0, 1: 0, 2: 0, 3: 0, 45: 1, 48: 2,
                    65: 2, 75: 3, 82: 3, 95: 3, 96: 4, 99: 4}


def _build_decision_features(pings, corridor, intersection, current_code):
    sev_code      = intersection.severity_code if intersection else current_code
    severity_num  = SEVERITY_NUMERIC.get(sev_code, 0)
    tti           = intersection.minutes_to_impact if intersection else 20
    time_norm     = min(1.0, tti / 20)
    p_int         = intersection.intersection_prob if intersection else 0.0
    road_info     = ROAD_TYPES.get(corridor.road_type, ROAD_TYPES['unknown'])
    road_score    = road_info['predictability']
    avg_accel     = sum(p.accel for p in pings[-3:]) / min(3, len(pings))
    is_braking    = 1 if avg_accel < -1.0 else 0
    curr_severe   = 1 if current_code in ALERT_THRESHOLD_CODES else 0
    speed_norm    = min(1.0, pings[-1].speed_mph / 75)
    return np.array([severity_num, time_norm, p_int, road_score,
                     is_braking, curr_severe, speed_norm], dtype=float)


print('✅ Phase 4A — Random Forest model trained!')


In [ ]:
# ─────────────────────────────────────────────────────────────────
# PHASE 4B — ALERT DECISION & OUTPUT ASSEMBLY  (FIXED)
# ─────────────────────────────────────────────────────────────────

ALERT_THRESHOLDS = {'RED': 0.70, 'YELLOW': 0.30}


def decide_alert(pings, corridor, intersection, weather_response):
    current_code, wind_speed, temperature_f = 0, 0.0, 70.0
    if weather_response:
        curr = weather_response.get('current', {})
        current_code  = curr.get('weather_code', 0)
        wind_speed    = curr.get('wind_speed_10m', 0.0)
        temperature_f = curr.get('temperature_2m', 70.0)

    current_condition = SEVERE_WEATHER_CODES.get(current_code, 'Unknown')

    features = _build_decision_features(pings, corridor, intersection, current_code)
    proba    = ALERT_MODEL.predict_proba(features.reshape(1, -1))[0]
    p_red    = proba[2] if len(proba) > 2 else 0.0
    p_yellow = proba[1] if len(proba) > 1 else 0.0
    p_impact = intersection.intersection_prob if intersection else 0.0

    severity_num  = features[0]
    time_norm_val = features[1]
    p_int_val     = features[2]

    # FIX: Hard override — tornado/heavy hail (severity>=4) within 15min -> RED
    # Prevents extreme-severity events being downgraded to YELLOW due to the
    # training-data gap in the moderate p_int (0.30-0.60) range.
    if severity_num >= 4 and time_norm_val <= 0.75 and p_int_val >= 0.20:
        alert_level = 'RED'
    elif p_red >= ALERT_THRESHOLDS['RED']:
        alert_level = 'RED'
    elif (p_red + p_yellow) >= ALERT_THRESHOLDS['YELLOW']:
        alert_level = 'YELLOW'
    elif current_code in ALERT_THRESHOLD_CODES:
        alert_level = 'RED' if ALERT_SEVERITY.get(current_code) in ('HIGH', 'CRITICAL') else 'YELLOW'
    else:
        alert_level = 'NONE'

    advice_code    = intersection.severity_code if intersection else current_code
    vehicle_advice = MOVING_VEHICLE_ADVICE.get(advice_code, '')
    exit_tip       = intersection.exit_before_impact if intersection else None

    if alert_level == 'RED':
        wx_label = intersection.weather_type if intersection else current_condition
        tti_str  = (f'in ~{int(intersection.minutes_to_impact)} min'
                    if intersection and intersection.minutes_to_impact > 0
                    else 'at your location NOW')
        alert_message = (
            f'🔴 RED ALERT [{ALERT_SEVERITY.get(advice_code, "HIGH")}]: '
            f'{wx_label} on your predicted path {tti_str}. '
            f'Impact probability: {p_impact*100:.0f}%. '
            f'Speed: {pings[-1].speed_mph:.0f} mph | Heading: {pings[-1].heading:.0f}°'
        )
    elif alert_level == 'YELLOW':
        wx_label = intersection.weather_type if intersection else current_condition
        tti_str  = (f'in ~{int(intersection.minutes_to_impact)} min'
                    if intersection and intersection.minutes_to_impact > 0
                    else 'near your location')
        alert_message = (
            f'🟡 YELLOW ADVISORY: {wx_label} possible on your path {tti_str}. '
            f'Impact probability: {p_impact*100:.0f}%. Monitor conditions.'
        )
    else:
        alert_message = '✅ No alert needed. Your predicted path is clear of severe weather.'

    return MovingVehicleAlert(
        current_lat=pings[-1].lat, current_lon=pings[-1].lon,
        speed_mph=round(pings[-1].speed_mph, 1), heading_degrees=round(pings[-1].heading, 1),
        road_type=corridor.road_type, corridor=corridor, intersection=intersection,
        current_weather=current_condition, current_code=current_code,
        wind_speed_mph=wind_speed, temperature_f=temperature_f,
        alert_level=alert_level, alert_message=alert_message,
        vehicle_advice=vehicle_advice, exit_tip=exit_tip,
        confidence_pct=round(p_impact * 100, 1),
    )


def run_phase4(pings, corridor, intersection, weather_response):
    return decide_alert(pings, corridor, intersection, weather_response)


print('✅ Phase 4B — Alert decision engine defined!')


---
## 🔗 Full Pipeline & Display

In [ ]:
# ─────────────────────────────────────────────────────────────────
# FULL PIPELINE ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────

def run_safe_pilot_moving(
    raw_pings: List[GPSPing],
    label: str = 'Vehicle',
    _mock_weather: Optional[Dict] = None,
    _mock_nws: Optional[List] = None,
) -> MovingVehicleAlert:
    """
    Main entry point: runs all 4 phases for a moving vehicle.

    Args:
        raw_pings:     List of 10–15 GPSPing objects (oldest → newest).
        label:         Human-readable vehicle/scenario name for display.
        _mock_weather: Optional injected Open-Meteo response (for simulation).
        _mock_nws:     Optional injected NWS polygon list (for simulation).
    Returns:
        MovingVehicleAlert — the fully-reasoned alert for this vehicle state.
    """
    print(f'\n🛣️  Running Safe Pilot for: {label}')
    print(f'   GPS pings: {len(raw_pings)} | '
          f'Lat: {raw_pings[-1].lat:.4f} | Lon: {raw_pings[-1].lon:.4f}')

    # Phase 1 — Feature engineering
    print('   ⚙️  Phase 1: Feature extraction...')
    pings = run_phase1(deepcopy(raw_pings))

    # Phase 2 — Trajectory prediction
    print('   🧠 Phase 2: LSTM trajectory prediction...')
    corridor = run_phase2(pings)

    # Phase 3 — Weather intersection
    print('   🌩️  Phase 3: Weather intersection...')
    if _mock_weather is not None or _mock_nws is not None:
        # Simulation mode: use injected data
        nws_polygons     = _mock_nws if _mock_nws is not None else []
        weather_response = _mock_weather
        intersection     = intersect_corridor_with_weather(
            corridor, nws_polygons, weather_response, pings
        )
    else:
        intersection, weather_response = run_phase3(pings, corridor)

    # Phase 4 — Alert decision
    print('   🎯 Phase 4: Alert decision...')
    alert = run_phase4(pings, corridor, intersection, weather_response)

    return alert


def print_moving_alert(alert: MovingVehicleAlert, label: str = ''):
    """
    Display a formatted MovingVehicleAlert — designed for glanceable in-car UX.
    High-contrast layout with the most critical info at the top.

    Args:
        alert: Populated MovingVehicleAlert from run_safe_pilot_moving().
        label: Optional scenario name for the header.
    """
    SEP  = '=' * 72
    DASH = '-' * 72

    print(SEP)
    if label:
        print(f'  🛡️  {label}')
    print(f'  📍 Position : {alert.current_lat:.4f}°N, {alert.current_lon:.4f}°W')
    print(f'  🚗 Speed    : {alert.speed_mph:.0f} mph  |  '
          f'Heading: {alert.heading_degrees:.0f}°  |  '
          f'Road: {alert.road_type}')
    print(DASH)

    # Current conditions
    print(f'  🌡️  Temp     : {alert.temperature_f:.1f} °F')
    print(f'  🌤️  Now      : {alert.current_weather} (code {alert.current_code})')
    print(f'  💨 Wind     : {alert.wind_speed_mph:.1f} mph')

    # Predicted path corridor
    print(DASH)
    print(f'  🧠 Predicted Path Corridor (next 20 min):')
    for wp in alert.corridor.waypoints:
        print(f'     +{wp.minutes_ahead:2d} min → '
              f'({wp.lat:.4f}°, {wp.lon:.4f}°)  '
              f'conf={wp.confidence:.2f}  '
              f'width=±{wp.corridor_radius_m:.0f}m')
    print(f'  📊 Corridor avg confidence: {alert.corridor.avg_confidence:.2f}')

    # Weather intersection
    print(DASH)
    if alert.intersection:
        ix = alert.intersection
        print(f'  ⚡ Weather on Path: {ix.weather_type}')
        print(f'     Time to Impact : ~{ix.minutes_to_impact:.0f} min')
        print(f'     P(Impact)      : {ix.intersection_prob*100:.0f}%')
    else:
        print('  ✅  No weather threats detected on predicted path.')

    # Alert decision
    print(DASH)
    alert_icon = {'RED': '🔴', 'YELLOW': '🟡', 'NONE': '✅'}.get(alert.alert_level, '❓')
    print(f'  {alert_icon} {alert.alert_message}')

    if alert.vehicle_advice:
        print(f'\n  💡 Driver Action: {alert.vehicle_advice}')

    if alert.exit_tip:
        print(f'  🛣️  Exit Guidance : {alert.exit_tip}')

    print(SEP)
    print()


print('✅ Pipeline orchestrator & display defined!')

---
## 🧰 Simulation Helpers

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SIMULATION HELPERS
# ─────────────────────────────────────────────────────────────────

def make_gps_trace(
    start_lat: float, start_lon: float,
    heading_deg: float, speed_mph: float,
    n_pings: int = 12, interval_sec: int = 5,
) -> List[GPSPing]:
    """
    Generate a synthetic GPS trace for a vehicle moving at constant
    speed and heading.  Adds small Gaussian noise to simulate real GPS jitter.

    Args:
        start_lat/lon:  Starting coordinate.
        heading_deg:    Direction of travel (0–360°, clockwise from North).
        speed_mph:      Speed of travel.
        n_pings:        Number of GPS samples to generate.
        interval_sec:   Seconds between each GPS ping.
    Returns:
        List of GPSPing objects ready for Phase 1 processing.
    """
    speed_ms   = speed_mph / MPH_PER_MS  # Convert to m/s
    dist_per_ping = speed_ms * interval_sec  # Metres per ping
    base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)

    pings = []
    lat, lon = start_lat, start_lon

    for i in range(n_pings):
        # Add GPS jitter (±5 m laterally, ±3 m along path)
        jitter_lat = np.random.normal(0, 0.00004)  # ~4 m lat jitter
        jitter_lon = np.random.normal(0, 0.00004)  # ~4 m lon jitter

        pings.append(GPSPing(
            lat=lat + jitter_lat,
            lon=lon + jitter_lon,
            alt=200.0 + np.random.normal(0, 1),
            timestamp=base_time + timedelta(seconds=i * interval_sec),
        ))

        # Advance position for next ping
        lat, lon = project_point(lat, lon, heading_deg, dist_per_ping)

    return pings


def make_mock_nws_polygon(
    center_lat: float, center_lon: float,
    radius_deg: float,
    event: str = 'Severe Thunderstorm Warning',
) -> List[Dict]:
    """
    Create a synthetic NWS warning polygon centered at a coordinate.
    Used in simulated scenarios to place a weather zone on the vehicle's path.

    Args:
        center_lat/lon: Center of the weather polygon.
        radius_deg:     Radius in decimal degrees (~0.1° ≈ 7 miles).
        event:          NWS event type string.
    Returns:
        List with one mock NWS polygon dict.
    """
    n_pts = 16
    pts = [
        (
            center_lon + radius_deg * math.cos(2 * math.pi * i / n_pts),
            center_lat + radius_deg * math.sin(2 * math.pi * i / n_pts),
        )
        for i in range(n_pts)
    ]
    polygon = Polygon(pts)
    code = NWS_EVENT_TO_CODE.get(event, 95)
    return [{'event': event, 'severity': 'Severe', 'polygon': polygon, 'code': code}]


def make_mock_weather_response(
    current_code: int = 0,
    wind_mph: float = 10.0,
    temp_f: float = 75.0,
    hourly_codes: Optional[List[int]] = None,
) -> Dict:
    """
    Build a minimal mock Open-Meteo API response for simulation.
    Injects specific weather codes without making a real API call.
    """
    codes = hourly_codes if hourly_codes else [current_code] * 24
    now   = datetime.utcnow()
    times = [(now + timedelta(hours=i)).strftime('%Y-%m-%dT%H:00') for i in range(24)]
    return {
        'current': {
            'weather_code':   current_code,
            'wind_speed_10m': wind_mph,
            'temperature_2m': temp_f,
            'precipitation':  0.0,
            'visibility':     24000,
        },
        'hourly': {
            'time':                     times,
            'weather_code':             codes,
            'precipitation_probability': [60 if c in ALERT_THRESHOLD_CODES else 5 for c in codes],
            'wind_gusts_10m':           [65.0 if c in ALERT_THRESHOLD_CODES else 10.0 for c in codes],
            'visibility':               [2000 if c in ALERT_THRESHOLD_CODES else 24000 for c in codes],
        },
    }


print('✅ Simulation helpers defined!')

---
## 🌍 Section A — Live Demo with Real GPS Traces

Simulates 3 vehicles driving through real US locations and checks actual
live weather and NWS alerts along their predicted paths.

> Results depend on actual current weather — alerts will vary by conditions at run time.

In [ ]:
np.random.seed(42)  # Reproducible GPS jitter

print('\n' + '🛡️  SAFE PILOT — SECTION A: Live Moving Vehicle Demo'.center(72))
print(f"{'Run at: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'):^72}\n")

# ── Live scenario definitions ──
LIVE_SCENARIOS = [
    {
        'label':       'Vehicle 1 — Oklahoma City, OK (Tornado Alley, Northbound I-35)',
        'start_lat':   35.4676, 'start_lon': -97.5164,
        'heading':     0,       # Northbound
        'speed_mph':   70,
    },
    {
        'label':       'Vehicle 2 — Miami, FL (Eastbound I-836, Hurricane Season)',
        'start_lat':   25.7617, 'start_lon': -80.3000,
        'heading':     90,      # Eastbound
        'speed_mph':   55,
    },
    {
        'label':       'Vehicle 3 — Denver, CO (Southbound I-25, Mountain Weather)',
        'start_lat':   39.7392, 'start_lon': -104.9903,
        'heading':     180,     # Southbound
        'speed_mph':   65,
    },
]

live_alerts = []
for scenario in LIVE_SCENARIOS:
    pings = make_gps_trace(
        start_lat   = scenario['start_lat'],
        start_lon   = scenario['start_lon'],
        heading_deg = scenario['heading'],
        speed_mph   = scenario['speed_mph'],
    )
    alert = run_safe_pilot_moving(pings, label=scenario['label'])
    print_moving_alert(alert, label=scenario['label'])
    live_alerts.append((scenario['label'], alert))

# Section A Summary
print('-' * 72)
print('📊 SECTION A SUMMARY')
print('-' * 72)
for label, alert in live_alerts:
    icon = {'RED': '🔴', 'YELLOW': '🟡', 'NONE': '✅'}.get(alert.alert_level, '❓')
    print(f'  {icon} {label}')
print('-' * 72)

---
## 🧪 Section B — Simulated Moving Vehicle Scenarios

These inject specific weather polygons and conditions so the demo
always produces deterministic, predictable alert output.

| # | Vehicle State | Weather on Path | Expected Alert |
|---|---|---|---|
| 1 | 70 mph highway, clear now | Tornado Warning polygon ahead | 🔴 RED |
| 2 | 65 mph highway, clear now | Thunderstorm + Hail in corridor | 🔴 RED |
| 3 | 55 mph, braking, residential | Flash Flood zone ahead | 🟡 YELLOW |
| 4 | 40 mph primary road, clear now | Heavy Rain in 15 min | 🟡 YELLOW |
| 5 | 70 mph highway, clear sky | No weather threats | ✅ NONE |

In [ ]:
np.random.seed(42)

print('\n' + '🧪  SAFE PILOT — SECTION B: Simulated Moving Scenarios'.center(72) + '\n')

# Base location: Edinburg, TX (warm, clear baseline)
BASE_LAT, BASE_LON = 26.3017, -98.1633

SIM_SCENARIOS = [
    # ── Scenario 1: Tornado Warning polygon placed ~8 miles ahead on highway ──
    {
        'label':       'Scenario 1 | 70 mph Highway → Tornado Warning in Path [RED]',
        'heading':     0, 'speed_mph': 70,
        'mock_weather': make_mock_weather_response(current_code=2, wind_mph=25, temp_f=82),
        'mock_nws':    make_mock_nws_polygon(
            center_lat=BASE_LAT + 0.18,  # ~12 miles north = within 20 min at 70 mph
            center_lon=BASE_LON,
            radius_deg=0.12,
            event='Tornado Warning',
        ),
    },
    # ── Scenario 2: Severe Thunderstorm Warning directly on predicted path ──
    {
        'label':       'Scenario 2 | 65 mph Highway → Thunderstorm + Hail Ahead [RED]',
        'heading':     45, 'speed_mph': 65,
        'mock_weather': make_mock_weather_response(current_code=3, wind_mph=18, temp_f=78),
        'mock_nws':    make_mock_nws_polygon(
            center_lat=BASE_LAT + 0.12,
            center_lon=BASE_LON + 0.12,
            radius_deg=0.10,
            event='Severe Thunderstorm Warning',
        ),
    },
    # ── Scenario 3: Flash Flood on residential road, driver braking ──
    {
        'label':       'Scenario 3 | 20 mph Residential → Flash Flood Zone [YELLOW]',
        'heading':     90, 'speed_mph': 20,
        'mock_weather': make_mock_weather_response(current_code=63, wind_mph=12, temp_f=72),
        'mock_nws':    make_mock_nws_polygon(
            center_lat=BASE_LAT,
            center_lon=BASE_LON + 0.08,  # ~5 miles east = within range but low prob
            radius_deg=0.06,
            event='Flash Flood Warning',
        ),
    },
    # ── Scenario 4: Open-Meteo forecasts heavy rain in 15 min ahead ──
    {
        'label':       'Scenario 4 | 40 mph Primary → Heavy Rain in 15 min [YELLOW]',
        'heading':     270, 'speed_mph': 40,
        'mock_weather': make_mock_weather_response(
            current_code=3, wind_mph=15, temp_f=68,
            hourly_codes=[3, 3, 65, 65, 65, 63, 1] + [0]*17,
        ),
        'mock_nws': [],  # No active NWS polygon
    },
    # ── Scenario 5: Clear sky, no threats anywhere ──
    {
        'label':       'Scenario 5 | 70 mph Highway → Clear Sky, No Alerts [NONE]',
        'heading':     180, 'speed_mph': 70,
        'mock_weather': make_mock_weather_response(current_code=0, wind_mph=8, temp_f=75),
        'mock_nws':    [],
    },
]

sim_results = []
for s in SIM_SCENARIOS:
    pings = make_gps_trace(
        start_lat=BASE_LAT, start_lon=BASE_LON,
        heading_deg=s['heading'], speed_mph=s['speed_mph'],
    )
    alert = run_safe_pilot_moving(
        pings,
        label=s['label'],
        _mock_weather=s['mock_weather'],
        _mock_nws=s['mock_nws'],
    )
    print_moving_alert(alert, label=s['label'])
    sim_results.append((s['label'], alert))

# Section B Summary
print('-' * 72)
print('📊 SECTION B SUMMARY')
print('-' * 72)
for label, alert in sim_results:
    icon = {'RED': '🔴', 'YELLOW': '🟡', 'NONE': '✅'}.get(alert.alert_level, '❓')
    print(f'  {icon} [{alert.alert_level:6s}] {label}')
print('-' * 72)

---
## ✏️ Section C — Test Your Own GPS Trace

Replace the values below with your actual GPS coordinates, heading, and speed.

**Tips:**
- Right-click any point on Google Maps → copy the lat/lon coordinates
- Heading 0° = North, 90° = East, 180° = South, 270° = West
- Use `live_mode=True` to pull real NWS + Open-Meteo data

In [ ]:
np.random.seed(0)

# ── Configure your vehicle state here ──
MY_LAT       = 32.7157   # Replace: your current latitude  (e.g. 40.7128 for NYC)
MY_LON       = -117.1611 # Replace: your current longitude (e.g. -74.0060 for NYC)
MY_HEADING   = 45        # Replace: direction of travel in degrees (0=N, 90=E, 180=S, 270=W)
MY_SPEED_MPH = 65        # Replace: your current speed in mph
LIVE_MODE    = True      # True = real NWS + Open-Meteo APIs; False = no weather APIs

# Generate synthetic GPS trace for your location
my_pings = make_gps_trace(
    start_lat   = MY_LAT,
    start_lon   = MY_LON,
    heading_deg = MY_HEADING,
    speed_mph   = MY_SPEED_MPH,
    n_pings     = 12,
)

if LIVE_MODE:
    # Real APIs — pulls live NWS alerts and Open-Meteo weather
    my_alert = run_safe_pilot_moving(
        my_pings,
        label='My Custom Vehicle',
    )
else:
    # Offline mode — inject a clear sky response
    my_alert = run_safe_pilot_moving(
        my_pings,
        label='My Custom Vehicle (Offline)',
        _mock_weather=make_mock_weather_response(current_code=1, wind_mph=10, temp_f=72),
        _mock_nws=[],
    )

print_moving_alert(my_alert, label='My Custom Vehicle')
my_alert  # Also return the object for inspection